In [2]:
# arabic_troll_vs_news.py
# -----------------------
# !pip install pandas tensorflow tensorflow-text==2.15  # first time only
import pandas as pd, tensorflow as tf, re, os
from tensorflow.keras import layers

MAX_LEN   = 120
VOCAB     = 25_000
BATCH     = 64
EMBED_DIM = 128
RNN_DIM   = 128
EPOCHS    = 4
SEED      = 42
tf.random.set_seed(SEED)

# 1. load -----------------------------------------------------------------
df = pd.read_csv('/content/fixed_file.csv')          # id  text  polarization
print(df.shape, df.polarization.value_counts())


(3380, 3) polarization
0    1868
1    1512
Name: count, dtype: int64


In [5]:
import numpy as np
# 2. very light clean (keep Arabic letters + spaces)
def clean(t):
    t = re.sub(r'[^ء-ي\s]', ' ', str(t))
    return ' '.join(t.split())[:400]          # truncate very long tweets
df['text'] = df.text.map(clean)

# 3. split 80/10/10
train, val, test = np.split(
        df.sample(frac=1, random_state=SEED),
        [int(.8*len(df)), int(.9*len(df))])

# 4. tf.data --------------------------------------------------------------
def ds_from_df(frame):
    return tf.data.Dataset.from_tensor_slices(
            (frame.text.values, frame.polarization.values)) \
          .shuffle(2000).batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = ds_from_df(train)
val_ds   = ds_from_df(val)
test_ds  = ds_from_df(test)

# 5. vectoriser ------------------------------------------------------------
vectoriser = layers.TextVectorization(
        max_tokens=VOCAB,
        standardize='lower',
        output_mode='int',
        output_sequence_length=MAX_LEN)
vectoriser.adapt(train_ds.map(lambda x,_: x))



/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [11]:
model = tf.keras.Sequential([
    vectoriser,                                    # (None,)  strings in
    layers.Embedding(VOCAB, EMBED_DIM, mask_zero=True,
                 embeddings_regularizer=tf.keras.regularizers.l2(1e-4)),
layers.Bidirectional(layers.GRU(RNN_DIM,
                                dropout=0.4,
                                recurrent_dropout=0.4)),
layers.Dense(64, activation='relu',
             kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
layers.Dropout(0.5),
layers.Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
model.summary()



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 120)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.fit(train_ds,
          validation_data=val_ds,
          epochs=EPOCHS)

# 8. final test ------------------------------------------------------------
print('Test accuracy:', model.evaluate(test_ds, verbose=0)[1])


Epoch 1/4
43/43 ━━━━━━━━━━━━━━━━━━━━ 41s 742ms/step - accuracy: 0.5399 - loss: 0.8743 - val_accuracy: 0.5592 - val_loss: 0.7165
Epoch 2/4
43/43 ━━━━━━━━━━━━━━━━━━━━ 30s 707ms/step - accuracy: 0.6584 - loss: 0.6265 - val_accuracy: 0.7278 - val_loss: 0.5701
Epoch 3/4
43/43 ━━━━━━━━━━━━━━━━━━━━ 31s 729ms/step - accuracy: 0.9654 - loss: 0.1696 - val_accuracy: 0.7071 - val_loss: 0.7691
Epoch 4/4
43/43 ━━━━━━━━━━━━━━━━━━━━ 30s 706ms/step - accuracy: 0.9969 - loss: 0.0617 - val_accuracy: 0.7012 - val_loss: 0.8800
Test accuracy: 0.7248520851135254


In [13]:
import numpy as np   # add at top

def predict(text: str):
    x = np.array([clean(text)])
    prob = float(model(x)[0, 0])
    return 'negative' if prob > 0.5 else 'neutral/positive', prob

print(predict('احلام انتي ونعالي ومنو انتي حتى تقيمين الفنانين'))
print(predict('ثلاث وفيات بحادث سير مروع في بابل'))

('negative', 0.8566731810569763)
('neutral/positive', 0.0039065987803041935)
